# No-GPU mode (live telemetry)

BeamNG.tech in **no-GPU mode** (`nogpu=True` → `-gfx null -headless`): no GPU, no window.

This notebook steps the sim and shows **live position / speed** in the cell output.

**Limits:** not for map-faithful driving — roads, cameras, lidar, and Flowgraph need a real render backend. See `headless_mode_camera_streaming.ipynb` for headless **with** GPU.

**Knobs** (setup cell): `HOST`, `PORT`, `LAUNCH`, `HISTORY`, etc.

Run cells in order. The live cell runs until you **interrupt** it (Stop). Then run the **cleanup** cell.

In [6]:
from __future__ import annotations

import math
import time

import matplotlib.pyplot as plt
from IPython.display import clear_output, display

from beamngpy import BeamNGpy, Scenario, Vehicle
from beamngpy.logging import BNGDisconnectedError

# --- knobs ---
HOST = "localhost"
PORT = 25252
# BNG_HOME = r"F:\BeamNG.tech.v0.38.5.0"
LAUNCH = True

LEVEL = "smallgrid"
WARMUP_STEPS = 5
HISTORY = 500  # max points kept in live plots
THROTTLE = 0.4
DISPLAY_INTERVAL_S = 0.05  # throttle matplotlib refresh (~20 Hz)


def speed_ms(vel: tuple[float, float, float] | list[float] | None) -> float:
    if vel is None:
        return 0.0
    return math.sqrt(vel[0] ** 2 + vel[1] ** 2 + vel[2] ** 2)


def fetch_vehicle_state(vehicle: Vehicle) -> tuple[tuple[float, float, float], tuple[float, float, float]]:
    vehicle.sensors.poll("state")
    pos = vehicle.state.get("pos") or (0.0, 0.0, 0.0)
    vel = vehicle.state.get("vel") or (0.0, 0.0, 0.0)
    return pos, vel

In [7]:
beamng = BeamNGpy(
    HOST,
    PORT,
    # home=BNG_HOME,
    nogpu=True,
    quit_on_close=True,
)

scenario = Scenario(LEVEL, "nogpu_basic_live")
ego = Vehicle("ego", model="etk800", license="NOGPU", color="White")
scenario.add_vehicle(ego)

beamng.open(launch=LAUNCH)
scenario.make(beamng)


In [8]:
beamng.settings.set_deterministic(60)
beamng.control.pause()
beamng.scenario.load(scenario)  # connects vehicles; do not call await_spawn after
beamng.scenario.start()

for _ in range(WARMUP_STEPS):
    beamng.control.step(1)

In [9]:
fig, (ax_xy, ax_speed) = plt.subplots(1, 2, figsize=(10, 4))
xs: list[float] = []
ys: list[float] = []
speeds: list[float] = []
frame_times: list[float] = []

ego.control(throttle=THROTTLE)
t0 = time.monotonic()
last_display = 0.0
step = 0
pos = (0.0, 0.0, 0.0)
vel = (0.0, 0.0, 0.0)

try:
    while True:
        frame_t0 = time.monotonic()
        beamng.control.step(1)
        pos, vel = fetch_vehicle_state(ego)
        spd = speed_ms(vel)

        xs.append(pos[0])
        ys.append(pos[1])
        speeds.append(spd)
        if len(xs) > HISTORY:
            xs.pop(0)
            ys.pop(0)
            speeds.pop(0)

        frame_times.append(time.monotonic() - frame_t0)
        if len(frame_times) > HISTORY:
            frame_times.pop(0)

        step += 1
        now = time.monotonic()
        if now - last_display < DISPLAY_INTERVAL_S:
            continue

        last_display = now
        elapsed = now - t0
        loop_hz = step / elapsed if elapsed > 0 else 0.0
        last_hz = 1.0 / frame_times[-1] if frame_times[-1] > 0 else 0.0

        ax_xy.clear()
        ax_xy.plot(xs, ys, "b-")
        ax_xy.scatter([xs[-1]], [ys[-1]], c="red", s=30)
        ax_xy.set_title("XY path")
        ax_xy.set_xlabel("x")
        ax_xy.set_ylabel("y")
        ax_xy.axis("equal")
        ax_xy.grid(True, alpha=0.3)

        ax_speed.clear()
        ax_speed.plot(speeds, "g-")
        ax_speed.set_title("Speed (m/s)")
        ax_speed.set_xlabel("step")
        ax_speed.grid(True, alpha=0.3)

        fig.suptitle(
            f"step {step}  |  pos {pos[0]:.1f}, {pos[1]:.1f}, {pos[2]:.1f}  |  speed {spd:.2f} m/s  |  loop {loop_hz:.1f} Hz  |  last {last_hz:.1f} Hz",
            y=1.02,
        )
        clear_output(wait=True)
        display(fig)
except KeyboardInterrupt:
    avg_hz = len(frame_times) / sum(frame_times) if frame_times else 0.0
    print(f"Stopped at step {step}")
    print(f"Final pos: {pos}")
    print(f"Final vel: {vel}")
    print(f"Measured loop rate: {avg_hz:.1f} Hz over {len(frame_times)} steps")
except BNGDisconnectedError:
    print("BeamNG disconnected — re-run the launch/load cells and try again.")
finally:
    plt.close(fig)

Stopped at step 1308
Final pos: [-2.9283758644014597, -436.68374437093735, 0.20366337103769183]
Final vel: [-0.280915230512619, -32.3109245300293, -0.0013958883937448263]
Measured loop rate: 35.5 Hz over 500 steps


In [10]:
beamng.disconnect()